In [9]:
import json
with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-1.5b.direct.step-2.full/train.6.9,15:53-5.json', 'r') as f:
    full_step2_data = json.load(f)
len(full_step2_data)

23325

## Raw Datasets for 5 samples

In [2]:
#1. upload raw step1 json file 
import json
from datasets import Dataset

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-1.5b.direct.step-1.full/train.6.9,4:29-5.json', 'r') as f:
    step1_full_raw = json.load(f)

dataset = Dataset.from_list(step1_full_raw)
dataset.push_to_hub('talzoomanzoo/0609-1.5B-step1-full-raw', private=True)

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  1.83ba/s]
/home/hojinkim/miniconda3/envs/py310/lib/python3.10/site-packages/huggingface_hub/lfs.py:337: UserWarning: hf_transfer is enabled but does not support uploading from bytes or BinaryIO, falling back to regular upload
  warnings.warn(
Uploading the dataset shards: 100%|██████████| 1/1 [00:08<00:00,  8.31s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0609-1.5B-step1-full-raw/commit/e00cbb38b512df64c7dace804676195df941c145', commit_message='Upload dataset', commit_description='', oid='e00cbb38b512df64c7dace804676195df941c145', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0609-1.5B-step1-full-raw', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0609-1.5B-step1-full-raw'), pr_revision=None, pr_num=None)

In [3]:
#2. upload raw step2 json file 
import json
from datasets import Dataset

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-1.5b.direct.step-2.full/train.6.9,15:53-5.json', 'r') as f:
    step2_full_raw = json.load(f)

step2_full_raw_filtered = []

for data in step2_full_raw:
    try:
        # Get the first available index (0-4) for each field
        question = next((data[f'Question_{i}'] for i in range(5) if f'Question_{i}' in data), None)
        answer = next((data[f'Answer_{i}'] for i in range(5) if f'Answer_{i}' in data), None)
        output = next((data[f'Output_{i}'] for i in range(5) if f'Output_{i}' in data), None)
        output_tokens = next((data[f'output_tokens_{i}'] for i in range(5) if f'output_tokens_{i}' in data), None)
        pred_answer = next((data[f'Pred_Answer_{i}'] for i in range(5) if f'Pred_Answer_{i}' in data), None)
        metrics = next((data[f'Metrics_{i}'] for i in range(5) if f'Metrics_{i}' in data), None)
        
        if all(v is not None for v in [question, answer, output, output_tokens, pred_answer, metrics]):
            step2_full_raw_filtered.append({
                'id': data['new_id'],
                'Question': question,
                'Answer': answer,
                'Output': output,
                'Output_tokens': output_tokens,
                'Pred_Answer': pred_answer,
                'Metrics': metrics
            })
    except Exception as e:
        print(f"Error processing item {data.get('new_id', 'unknown')}: {str(e)}")
        continue

print(f"Processed {len(step2_full_raw_filtered)} items successfully")

dataset = Dataset.from_list(step2_full_raw_filtered)
dataset.push_to_hub('talzoomanzoo/0609-1.5B-step2-full-raw', private=True)

Processed 23325 items successfully


Creating parquet from Arrow format: 100%|██████████| 12/12 [00:00<00:00, 37.46ba/s]
/home/hojinkim/miniconda3/envs/py310/lib/python3.10/site-packages/huggingface_hub/lfs.py:337: UserWarning: hf_transfer is enabled but does not support uploading from bytes or BinaryIO, falling back to regular upload
  warnings.warn(
Uploading the dataset shards: 100%|██████████| 2/2 [00:10<00:00,  5.28s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0609-1.5B-step2-full-raw/commit/39a0f8e73ca0523b3347c3ca669fa898022edd31', commit_message='Upload dataset', commit_description='', oid='39a0f8e73ca0523b3347c3ca669fa898022edd31', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0609-1.5B-step2-full-raw', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0609-1.5B-step2-full-raw'), pr_revision=None, pr_num=None)

## RFT Dataset

In [10]:
#1. rft-y-ytrue

import json
import random

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-1.5b.direct.step-2.full/train.6.9,15:53-5.json', 'r') as f:
    rft_data = json.load(f)

rft_y_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

# First, separate data into math_equal=True and math_equal=False groups
true_examples = []
false_examples = []

for data in rft_data:
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    entry = {
        'idx': len(true_examples if has_math_equal else false_examples) + 1,
        'id': data.get('new_id', None),
        'input_x': '',
        'ground_truth': '',
        'output_z': '',
        'output_y': '',
        'metrics': []
    }

    # Collect all matching fields
    for key in data:
        if key.startswith('Question_'):
            entry['input_x'] = extract_x(data[key])
            entry['output_z'] = "<think>" + extract_z(data[key])
        elif key.startswith('Answer_'):
            entry['ground_truth'] = data[key]
        elif key.startswith('Output_'):
            entry['output_y'] = "</think>" + data[key]
        elif key.startswith('Metrics_'):
            entry['metrics'].append(data[key])
    
    if has_math_equal:
        true_examples.append(entry)
    else:
        false_examples.append(entry)

# Create pairs of examples
count = 0
for true_ex in true_examples:
    for false_ex in false_examples:
        if true_ex['input_x'] + true_ex['output_z'] == false_ex['input_x'] + false_ex['output_z']:  # Only pair examples with same input
            count += 1
            rft_y_entry = {
                'idx': count,
                'id': f"{true_ex['id']}_{false_ex['id']}",
                'input_x_z': true_ex['input_x'] + true_ex['output_z'],
                'ground_truth': true_ex['ground_truth'],
                'chosen_y': true_ex['output_y'],
                'rejected_y': false_ex['output_y'],
                'metrics': true_ex['metrics'] + false_ex['metrics']
            }
            rft_y_data.append(rft_y_entry)

random.seed(42)
random.shuffle(rft_y_data)
rft_y_data = rft_y_data[:1000]

with open('0609_1.5B_rft_y_data.json', 'w') as f:
    json.dump(rft_y_data, f, ensure_ascii=False, indent=4)
print(f"Created {len(rft_y_data)} RFT pairs")
print(f"Total true examples: {len(true_examples)}")
print(f"Total false examples: {len(false_examples)}")

Created 1000 RFT pairs
Total true examples: 6021
Total false examples: 17304


In [11]:
from datasets import Dataset

dataset = Dataset.from_list(rft_y_data)
dataset.push_to_hub('talzoomanzoo/0609_1.5B_rft_y_ytrue', private=True)

/home/hojinkim/miniconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.88s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0609_1.5B_rft_y_ytrue/commit/372361f597dd7037782a8ae28baaf92e6c76931e', commit_message='Upload dataset', commit_description='', oid='372361f597dd7037782a8ae28baaf92e6c76931e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0609_1.5B_rft_y_ytrue', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0609_1.5B_rft_y_ytrue'), pr_revision=None, pr_num=None)

In [12]:
#2. rft-zy-ytrue

import json
import random

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-1.5b.direct.step-2.full/train.6.9,15:53-5.json', 'r') as f:
    rft_data = json.load(f)

rft_z_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

# First, separate data into math_equal=True and math_equal=False groups
true_examples = []
false_examples = []

for data in rft_data:
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    entry = {
        'idx': len(true_examples if has_math_equal else false_examples) + 1,
        'id': data.get('new_id', None),
        'input_x': '',
        'ground_truth': '',
        'output_z': '',
        'output_y': '',
        'metrics': []
    }

    # Collect all matching fields
    for key in data:
        if key.startswith('Question_'):
            entry['input_x'] = extract_x(data[key])
            entry['output_z'] = "<think>" + extract_z(data[key])
        elif key.startswith('Answer_'):
            entry['ground_truth'] = data[key]
        elif key.startswith('Output_'):
            entry['output_y'] = "</think>" + data[key]
        elif key.startswith('Metrics_'):
            entry['metrics'].append(data[key])
    
    if has_math_equal:
        true_examples.append(entry)
    else:
        false_examples.append(entry)

# Create pairs of examples
count = 0
for true_ex in true_examples:
    count += 1
    rft_entry = {
                'idx': count,
                'id': f"{true_ex['id']}",
                'input_x': true_ex['input_x'],
                'ground_truth': true_ex['ground_truth'],
                'chosen_z': true_ex['output_z'],
                'chosen_y': true_ex['output_y'],
                'chosen_zy': true_ex['output_z'] + true_ex['output_y'],
                'metrics': true_ex['metrics']
            }
    rft_z_data.append(rft_entry)

random.seed(42)
random.shuffle(rft_z_data)
rft_z_data = rft_z_data[:1000]

with open('0609_1.5B_rft_z_data.json', 'w') as f:
    json.dump(rft_z_data, f, ensure_ascii=False, indent=4)
print(f"Created {len(rft_z_data)} RFT pairs")
print(f"Total true examples: {len(true_examples)}")
print(f"Total false examples: {len(false_examples)}")


Created 1000 RFT pairs
Total true examples: 6021
Total false examples: 17304


In [13]:
from datasets import Dataset

dataset = Dataset.from_list(rft_y_data)
dataset.push_to_hub('talzoomanzoo/0609_1.5B_rft_zy_ytrue', private=True)

Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.15s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0609_1.5B_rft_zy_ytrue/commit/f8cd38b712b04afc434f2a76b439e8f2e98a38a9', commit_message='Upload dataset', commit_description='', oid='f8cd38b712b04afc434f2a76b439e8f2e98a38a9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0609_1.5B_rft_zy_ytrue', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0609_1.5B_rft_zy_ytrue'), pr_revision=None, pr_num=None)

In [14]:
#3. rft-zy-zthreshold
import json
import re

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-1.5b.direct.step-2.full/train.6.9,15:53-5.json', 'r') as f:
    z_y_combined_data = json.load(f)

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

filtered_z_y_combined_data = []

# Process data in chunks of 25 rows
for idx in range(0, len(z_y_combined_data), 25):
    chunk_25 = z_y_combined_data[idx:idx+25]
    subgroup_data = []  # Store (percentage, data) pairs for each subgroup
    subgroup_z_chosen_y_chosen = []
    subgroup_z_chosen_y_rejected = []
    subgroup_z_rejected_y_chosen = []
    subgroup_z_rejected_y_rejected = []
    
    # Process each subgroup of 5 rows within the 25-row chunk
    for i in range(0, len(chunk_25), 5):
        subgroup = chunk_25[i:i+5]
        true_pairs = []
        false_pairs = []
        
        # Count true and false pairs in this subgroup
        for data in subgroup:
            for key in data:
                if key.startswith('Metrics_'):
                    if data[key].get('math_equal', True):
                        true_pairs.append(data)
                    else:
                        false_pairs.append(data)
        
        # Calculate z_true_percentage for this subgroup
        z_true_percentage = len(true_pairs) / len(subgroup)
        
        # Store the percentage and corresponding data
        if 0.5 <= z_true_percentage < 1.0 and true_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'chosen'))
        elif 0 < z_true_percentage < 0.5 and false_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'rejected'))
    
    # If we have data, find the highest and lowest percentage entries
    if subgroup_data:
        # Sort by percentage
        subgroup_data.sort(key=lambda x: x[0])
        
        # Get the highest percentage (chosen) and lowest percentage (rejected)
        chosen_z_data = None
        rejected_z_data = None
        chosen_z_chosen_y_data = None
        chosen_z_rejected_y_data = None
        rejected_z_chosen_y_data = None
        rejected_z_rejected_y_data = None
        
        for percentage, true_data, false_data, type_ in subgroup_data:
            if type_ == 'chosen' and chosen_z_data is None:
                chosen_z_data = true_data
                chosen_z_chosen_y_data = true_data
                chosen_z_rejected_y_data = false_data
            elif type_ == 'rejected' and rejected_z_data is None:
                rejected_z_data = false_data
                rejected_z_chosen_y_data = true_data
                rejected_z_rejected_y_data = false_data
        
        # If we have both chosen and rejected data, create an entry
        if chosen_z_data and rejected_z_data:
            #x, z_chosen, y_chosen under z _chosen, y_reject under z _chosen, z_reject, y_chosen under z _reject, y_reject under z _reject 
            entry = {
                'idx': idx+1,
                'id': chosen_z_data.get('new_id', None).split('_')[0],
                'input_x': '',
                'chosen_z': '',
                'chosen_zy': '',
                'chosen_y_under_chosen_z': '',
                'rejected_y_under_chosen_z': '',
                'rejected_z': '',
                'rejected_zy': '',
                'chosen_y_under_rejected_z': '',
                'rejected_y_under_rejected_z': ''
            }
            
            # Process chosen z data
            for key in chosen_z_data:
                if key.startswith('Question_'):
                    entry['input_x'] = extract_x(chosen_z_data[key])
                    entry['chosen_z'] = "<think>" + extract_z(chosen_z_data[key])
                if key.startswith('Answer_'):
                    entry['ground_truth'] = chosen_z_data[key]
            for key in chosen_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_chosen_z'] = "</think>" + chosen_z_chosen_y_data[key]
                    entry['chosen_zy'] = entry['chosen_z'] + entry['chosen_y_under_chosen_z']
            for key in chosen_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_chosen_z'] = "</think>" + chosen_z_rejected_y_data[key]
            
            # Process rejected z data
            for key in rejected_z_data:
                if key.startswith('Question_'):
                    entry['rejected_z'] = "<think>" + extract_z(rejected_z_data[key])
            for key in rejected_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_rejected_z'] = "</think>" + rejected_z_chosen_y_data[key]
            for key in rejected_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_rejected_z'] = "</think>" + rejected_z_rejected_y_data[key]
                    entry['rejected_zy'] = entry['rejected_z'] + entry['rejected_y_under_rejected_z']
            filtered_z_y_combined_data.append(entry)

with open('0609-zy-combined-data.json', 'w') as f:
    json.dump(filtered_z_y_combined_data, f, ensure_ascii=False, indent=4)

print(len(filtered_z_y_combined_data))

396


In [15]:
from datasets import Dataset

dataset = Dataset.from_list(filtered_z_y_combined_data)
dataset.push_to_hub('talzoomanzoo/0609_1.5B_rft_zy_zthreshold', private=True)

Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.35s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0609_1.5B_rft_zy_zthreshold/commit/4d7e3ce9218635211e303096ffd373c06732474d', commit_message='Upload dataset', commit_description='', oid='4d7e3ce9218635211e303096ffd373c06732474d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0609_1.5B_rft_zy_zthreshold', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0609_1.5B_rft_zy_zthreshold'), pr_revision=None, pr_num=None)

## DPO Dataset

In [16]:
#1. dpo-y-ytrue
#4. 0603_dpo_y_ytrue
import json
import random

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-1.5b.direct.step-2.full/train.6.9,15:53-5.json', 'r') as f:
    dpo_data = json.load(f)

dpo_y_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

# First, separate data into math_equal=True and math_equal=False groups
true_examples = []
false_examples = []

for data in dpo_data:
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    entry = {
        'idx': len(true_examples if has_math_equal else false_examples) + 1,
        'id': data.get('new_id', None),
        'input_x': '',
        'ground_truth': '',
        'output_z': '',
        'output_y': '',
        'metrics': []
    }

    # Collect all matching fields
    for key in data:
        if key.startswith('Question_'):
            entry['input_x'] = extract_x(data[key])
            entry['output_z'] = "<think>" + extract_z(data[key])
        elif key.startswith('Answer_'):
            entry['ground_truth'] = data[key]
        elif key.startswith('Output_'):
            entry['output_y'] = "</think>" + data[key]
        elif key.startswith('Metrics_'):
            entry['metrics'].append(data[key])
    
    if has_math_equal:
        true_examples.append(entry)
    else:
        false_examples.append(entry)

# Create pairs of examples
count = 0
for true_ex in true_examples:
    for false_ex in false_examples:
        if true_ex['input_x'] + true_ex['output_z'] == false_ex['input_x'] + false_ex['output_z']:  # Only pair examples with same input
            count += 1
            dpo_entry = {
                'idx': count,
                'id': f"{true_ex['id']}_{false_ex['id']}",
                'input_x_z': true_ex['input_x'] + true_ex['output_z'],
                'ground_truth': true_ex['ground_truth'],
                'output_z': true_ex['output_z'],
                'chosen_y': true_ex['output_y'],
                'rejected_y': false_ex['output_y'],
                'metrics': true_ex['metrics'] + false_ex['metrics']
            }
            dpo_y_data.append(dpo_entry)

random.seed(42)
random.shuffle(dpo_y_data)
dpo_y_data = dpo_y_data[:1000]

with open('0609_1.5B_dpo_y_data.json', 'w') as f:
    json.dump(dpo_y_data, f, ensure_ascii=False, indent=4)
print(f"Created {len(dpo_y_data)} DPO pairs")
print(f"Total true examples: {len(true_examples)}")
print(f"Total false examples: {len(false_examples)}")


Created 1000 DPO pairs
Total true examples: 6021
Total false examples: 17304


In [17]:
#huggingface push dpo-y-ytrue
from datasets import Dataset

dataset = Dataset.from_list(dpo_y_data)
dataset.push_to_hub('talzoomanzoo/0609_1.5B_dpo-y-ytrue', private=True)

Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.98s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0609_1.5B_dpo-y-ytrue/commit/8383365c6993daa8443cdf036535ef3a1a9a2d4a', commit_message='Upload dataset', commit_description='', oid='8383365c6993daa8443cdf036535ef3a1a9a2d4a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0609_1.5B_dpo-y-ytrue', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0609_1.5B_dpo-y-ytrue'), pr_revision=None, pr_num=None)

In [18]:
#2. dpo-zy-ytrue
import json
import random

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-1.5b.direct.step-2.full/train.6.9,15:53-5.json', 'r') as f:
    dpo_data = json.load(f)

dpo_z_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

# First, separate data into math_equal=True and math_equal=False groups
true_examples = []
false_examples = []

for data in dpo_data:
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    entry = {
        'idx': len(true_examples if has_math_equal else false_examples) + 1,
        'id': data.get('new_id', None),
        'input_x': '',
        'ground_truth': '',
        'output_z': '',
        'output_y': '',
        'metrics': []
    }

    # Collect all matching fields
    for key in data:
        if key.startswith('Question_'):
            entry['input_x'] = extract_x(data[key])
            entry['output_z'] = "<think>" + extract_z(data[key])
        elif key.startswith('Answer_'):
            entry['ground_truth'] = data[key]
        elif key.startswith('Output_'):
            entry['output_y'] = "</think>" + data[key]
        elif key.startswith('Metrics_'):
            entry['metrics'].append(data[key])
    
    if has_math_equal:
        true_examples.append(entry)
    else:
        false_examples.append(entry)

# Create pairs of examples
count = 0
for true_ex in true_examples:
    for false_ex in false_examples:
        if true_ex['input_x'] == false_ex['input_x']:  # Only pair examples with same input
            count += 1
            dpo_entry = {
                'idx': count,
                'id': f"{true_ex['id']}_{false_ex['id']}",
                'input_x': true_ex['input_x'],
                'ground_truth': true_ex['ground_truth'],
                'chosen_z': true_ex['output_z'],
                'rejected_z': false_ex['output_z'],
                'chosen_y': true_ex['output_y'],
                'rejected_y': false_ex['output_y'],
                'chosen_zy': true_ex['output_z'] + true_ex['output_y'],
                'rejected_zy': false_ex['output_z'] + false_ex['output_y'],
                'metrics': true_ex['metrics'] + false_ex['metrics']
            }
            dpo_z_data.append(dpo_entry)

random.seed(42)
random.shuffle(dpo_z_data)
dpo_z_data = dpo_z_data[:1000]

with open('0609_1.5B_dpo_z_data.json', 'w') as f:
    json.dump(dpo_z_data, f, ensure_ascii=False, indent=4)
print(f"Created {len(dpo_z_data)} DPO pairs")
print(f"Total true examples: {len(true_examples)}")
print(f"Total false examples: {len(false_examples)}")



Created 1000 DPO pairs
Total true examples: 6021
Total false examples: 17304


In [19]:
from datasets import Dataset

dataset = Dataset.from_list(dpo_z_data)
dataset.push_to_hub('talzoomanzoo/0609_1.5B_rft_zy_ytrue', private=True, split="train")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  3.45ba/s]
/home/hojinkim/miniconda3/envs/py310/lib/python3.10/site-packages/huggingface_hub/lfs.py:337: UserWarning: hf_transfer is enabled but does not support uploading from bytes or BinaryIO, falling back to regular upload
  warnings.warn(
Uploading the dataset shards: 100%|██████████| 1/1 [00:05<00:00,  5.27s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0609_1.5B_rft_zy_ytrue/commit/f89bef41fac905c95a643a514e39d9ccf8f8302e', commit_message='Upload dataset', commit_description='', oid='f89bef41fac905c95a643a514e39d9ccf8f8302e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0609_1.5B_rft_zy_ytrue', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0609_1.5B_rft_zy_ytrue'), pr_revision=None, pr_num=None)

In [20]:
#3. dpo-zy-zthreshold

#3. dpo-zy-zthreshold
import json
import re

with open('/scratch/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-1.5b.direct.step-2.full/train.6.9,15:53-5.json', 'r') as f:
    z_y_combined_data = json.load(f)

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

filtered_z_y_combined_data = []

# Process data in chunks of 25 rows
for idx in range(0, len(z_y_combined_data), 25):
    chunk_25 = z_y_combined_data[idx:idx+25]
    subgroup_data = []  # Store (percentage, data) pairs for each subgroup
    subgroup_z_chosen_y_chosen = []
    subgroup_z_chosen_y_rejected = []
    subgroup_z_rejected_y_chosen = []
    subgroup_z_rejected_y_rejected = []
    
    # Process each subgroup of 5 rows within the 25-row chunk
    for i in range(0, len(chunk_25), 5):
        subgroup = chunk_25[i:i+5]
        true_pairs = []
        false_pairs = []
        
        # Count true and false pairs in this subgroup
        for data in subgroup:
            for key in data:
                if key.startswith('Metrics_'):
                    if data[key].get('math_equal', True):
                        true_pairs.append(data)
                    else:
                        false_pairs.append(data)
        
        # Calculate z_true_percentage for this subgroup
        z_true_percentage = len(true_pairs) / len(subgroup)
        
        # Store the percentage and corresponding data
        if 0.4 <= z_true_percentage < 1.0 and true_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'chosen'))
        elif 0 < z_true_percentage < 0.4 and false_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'rejected'))
    
    # If we have data, find the highest and lowest percentage entries
    if subgroup_data:
        # Sort by percentage
        subgroup_data.sort(key=lambda x: x[0])
        
        # Get the highest percentage (chosen) and lowest percentage (rejected)
        chosen_z_data = None
        rejected_z_data = None
        chosen_z_chosen_y_data = None
        chosen_z_rejected_y_data = None
        rejected_z_chosen_y_data = None
        rejected_z_rejected_y_data = None
        
        for percentage, true_data, false_data, type_ in subgroup_data:
            if type_ == 'chosen' and chosen_z_data is None:
                chosen_z_data = true_data
                chosen_z_chosen_y_data = true_data
                chosen_z_rejected_y_data = false_data
            elif type_ == 'rejected' and rejected_z_data is None:
                rejected_z_data = false_data
                rejected_z_chosen_y_data = true_data
                rejected_z_rejected_y_data = false_data
        
        # If we have both chosen and rejected data, create an entry
        if chosen_z_data and rejected_z_data:
            #x, z_chosen, y_chosen under z _chosen, y_reject under z _chosen, z_reject, y_chosen under z _reject, y_reject under z _reject 
            entry = {
                'idx': idx+1,
                'id': chosen_z_data.get('new_id', None).split('_')[0],
                'input_x': '',
                'chosen_z': '',
                'chosen_y_under_chosen_z': '',
                'rejected_y_under_chosen_z': '',
                'rejected_z': '',
                'chosen_y_under_rejected_z': '',
                'rejected_y_under_rejected_z': '',
                'chosen_z_y': '',
                'rejected_z_y': ''
            }
            
            # Process chosen z data
            for key in chosen_z_data:
                if key.startswith('Question_'):
                    entry['input_x'] = extract_x(chosen_z_data[key])
                    entry['chosen_z'] = "<think>" + extract_z(chosen_z_data[key])
                if key.startswith('Answer_'):
                    entry['ground_truth'] = chosen_z_data[key]
            for key in chosen_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_chosen_z'] = "</think>" + chosen_z_chosen_y_data[key]
                    entry['chosen_z_y'] = entry['chosen_z'] + entry['chosen_y_under_chosen_z']
            for key in chosen_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_chosen_z'] = "</think>" + chosen_z_rejected_y_data[key]
            
            # Process rejected z data
            for key in rejected_z_data:
                if key.startswith('Question_'):
                    entry['rejected_z'] = "<think>" + extract_z(rejected_z_data[key])
            for key in rejected_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_rejected_z'] = "</think>" + rejected_z_chosen_y_data[key]
            for key in rejected_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_rejected_z'] = "</think>" + rejected_z_rejected_y_data[key]
                    entry['rejected_z_y'] = entry['rejected_z'] + entry['rejected_y_under_rejected_z']
            filtered_z_y_combined_data.append(entry)

with open('0609-zy-combined-data.json', 'w') as f:
    json.dump(filtered_z_y_combined_data, f, ensure_ascii=False, indent=4)

print(len(filtered_z_y_combined_data))

507


In [21]:
#huggingface push dpo-zy-zthreshold
from datasets import Dataset

dataset = Dataset.from_list(filtered_z_y_combined_data)
dataset.push_to_hub('talzoomanzoo/0609_1.5B_dpo-zy-zthreshold', private=True)

Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.54s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0609_1.5B_dpo-zy-zthreshold/commit/8f585222cda82d2bdb8e7a1d04719c1b31e5a4c8', commit_message='Upload dataset', commit_description='', oid='8f585222cda82d2bdb8e7a1d04719c1b31e5a4c8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0609_1.5B_dpo-zy-zthreshold', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0609_1.5B_dpo-zy-zthreshold'), pr_revision=None, pr_num=None)